# ComfyUI on AMD Radeon Cloud

This notebook sets up ComfyUI with support for:
- **Wan2.2** (Video Generation)
- **Z-Image-Turbo** (Fast Image Generation)
- **Qwen-Edit** (Image Editing)

Running on AMD GPU with ROCm 6.2.4

In [ ]:
# Install system dependencies
!apt-get update && apt-get install -y git wget curl libgl1 libglib2.0-0 ffmpeg

In [ ]:
# Install PyTorch with ROCm 6.2.4 support
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/rocm6.2.4

In [ ]:
# Clone ComfyUI and ComfyUI-Manager
import os

os.chdir('/workspace')

!git clone https://github.com/comfyanonymous/ComfyUI.git
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git ComfyUI/custom_nodes/ComfyUI-Manager

os.chdir('/workspace/ComfyUI')
!pip install -r requirements.txt

In [ ]:
# Clone custom nodes for Wan2.2 video and other models
os.chdir('/workspace/ComfyUI/custom_nodes')

!git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git
!git clone https://github.com/kijai/ComfyUI-KJNodes.git

# Install extra dependencies
!pip install huggingface-hub modelscope accelerate safetensors einops av
!pip install -r ComfyUI-WanVideoWrapper/requirements.txt || true
!pip install -r ComfyUI-KJNodes/requirements.txt || true

In [ ]:
# Download Wan2.2 model from ModelScope (faster in China)
from modelscope.hub.snapshot_download import snapshot_download

model_dir = snapshot_download(
    'Wan-AI/Wan2.1-I2V-14B-480P',
    cache_dir='/workspace/ComfyUI/models/diffusion_models/Wan2.2'
)
print(f'Wan2.2 model downloaded to: {model_dir}')

In [ ]:
# Download CLIP and VAE models from HuggingFace (Comfy-Org repackaged)
from huggingface_hub import hf_hub_download

# CLIP text encoder
clip_path = hf_hub_download(
    repo_id='Comfy-Org/Wan_2.1_ComfyUI_repackaged',
    filename='split_files/text_encoders/umt5_xxl_fp8.safetensors',
    local_dir='/workspace/ComfyUI/models'
)
print(f'CLIP downloaded to: {clip_path}')

# VAE
vae_path = hf_hub_download(
    repo_id='Comfy-Org/Wan_2.1_ComfyUI_repackaged',
    filename='split_files/vae/wan_2.1_vae.safetensors',
    local_dir='/workspace/ComfyUI/models'
)
print(f'VAE downloaded to: {vae_path}')

In [ ]:
# Launch ComfyUI in a background thread
import threading
import subprocess

os.chdir('/workspace/ComfyUI')

def run_comfyui():
    proc = subprocess.Popen(
        ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188', '--force-fp16'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        print(line, end='')

thread = threading.Thread(target=run_comfyui, daemon=True)
thread.start()

import time
time.sleep(5)
print('ComfyUI is starting up...')
print('Access it at: http://<your-instance-ip>:8188')